**Set environment**

In [1]:
source ../run_config_project.sh
show_env

BASE DIRECTORY (FD_BASE):      /hpc/group/igvf/kk319
REPO DIRECTORY (FD_REPO):      /hpc/group/igvf/kk319/repo
WORK DIRECTORY (FD_WORK):      /hpc/group/igvf/kk319/work
DATA DIRECTORY (FD_DATA):      /hpc/group/igvf/kk319/data
CONTAINER DIR. (FD_SING):      /hpc/group/igvf/kk319/container

You are working with           
PATH OF PROJECT (FD_PRJ):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR
PROJECT RESULTS (FD_RES):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results
PROJECT SCRIPTS (FD_EXE):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts
PROJECT DATA    (FD_DAT):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/data
PROJECT NOTE    (FD_NBK):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/notebooks
PROJECT DOCS    (FD_DOC):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/docs
PROJECT LOG     (FD_LOG):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/log
PROJECT REF     (FD_REF):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/references
PR

## Preview

In [2]:
ls -1 ${FD_RES}

analysis_variant_motif_richard
analysis_variant_motif_richard_arc251231
analysis_variant_motif_richard_arc260223
predict_variant_alphagenome
predict_variant_kircher2019


In [3]:
ls ${FD_RES}/analysis_variant_motif_richard

background_zero_order.npy
background_zero_order.tsv
batches_dev
batches_pilot_low_dinuc
batches_pilot_low_ori
batches_pilot_top_dinuc
batches_pilot_top_ori
batches_top_dinuc
batches_top_ori
motifdelta_pilot_low_dinuc_jvierstra_v2.1beta
motifdelta_pilot_low_ori_jvierstra_v2.1beta
motifdelta_pilot_top_dinuc_jvierstra_v2.1beta
motifdelta_pilot_top_ori_jvierstra_v2.1beta
motifdelta_top_ori_jaspar2024
motifdelta_top_ori_jvierstra_v2.1beta
motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl
motif_jaspar2024_core_vertebrates_nonredundant.pmap.pkl
motif_jaspar2024_core_vertebrates_nonredundant.tbind.pkl
motif_nonredundant_jvierstra_v2.1beta.lods.pkl
motif_nonredundant_jvierstra_v2.1beta.pmap.pkl
motif_nonredundant_jvierstra_v2.1beta.tbind.pkl
motifscan_pilot_low_dinuc_jvierstra_v2.1beta
motifscan_pilot_low_ori_jvierstra_v2.1beta
motifscan_pilot_top_dinuc_jvierstra_v2.1beta
motifscan_pilot_top_ori_jvierstra_v2.1beta
motifscan_top_ori_jaspar2024
motifscan_top_ori_jvierstra_v2.1beta
variant_c

**Set compute resource**

In [4]:
### choose compute profile
CHOOSE_PARTITION="biostat"
#CHOOSE_PARTITION="igvf"
#CHOOSE_PARTITION="igvf_common"
#CHOOSE_PARTITION="biostat_common"

case "$CHOOSE_PARTITION" in
    igvf)
        SLURM_ACCOUNT="majoroslab"
        SLURM_PARTITION="igvf"
        ;;
    igvf_common)
        SLURM_ACCOUNT="majoroslab"
        SLURM_PARTITION="igvf,common"
        ;;
    biostat)
        SLURM_ACCOUNT="biostat"
        SLURM_PARTITION="biostat"
        ;;
    biostat_common)
        SLURM_ACCOUNT="biostat"
        SLURM_PARTITION="biostat,common"
        ;;
    *)
        echo "Unknown CHOOSE_PARTITION: $CHOOSE_PARTITION" >&2
        exit 1
        ;;
esac

### optional: node excludes (ONLY if needed)
EXCLUDE_COMMON="dcc-comp-10,dcc-core-08,dcc-core-53"
EXCLUDE_NODES=""

# Check if "common" is one of the comma-separated partitions
if [[ ",${SLURM_PARTITION}," == *",common,"* ]]; then
  EXCLUDE_NODES="${EXCLUDE_COMMON}"
fi

echo "SLURM_ACCOUNT=${SLURM_ACCOUNT}"
echo "SLURM_PARTITION=${SLURM_PARTITION}"
echo "EXCLUDE_NODES=${EXCLUDE_NODES:-<none>}"

SLURM_ACCOUNT=biostat
SLURM_PARTITION=biostat
EXCLUDE_NODES=<none>


## Dinucleotide Shuffling

### Prepare

In [5]:
ls -1 ${FD_RES}/analysis_variant_motif_richard/batches_top_ori

variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.top01k.ref.fa.gz
variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.top10k.ref.fa.gz


## Execute

In [6]:
### set script
FP_EXE=${FD_EXE}/run_sequence_dinuc_shuffle.sh

### set slurm opt
NUM_CPU=2
NUM_MEM=8G

LST_OPTS=(
    -A "${SLURM_ACCOUNT}"
    -p "${SLURM_PARTITION}"
    --cpus-per-task="${NUM_CPU}"
    --mem="${NUM_MEM}"
    --chdir="${FD_EXE}"
    --export=ALL,FP_CNF="${FP_CNF}"
    --parsable
)

if [[ -n "${EXCLUDE_NODES}" ]]; then
  LST_OPTS+=( --exclude="${EXCLUDE_NODES}" )
fi

### set parameters
FN_PREFIX="variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs"
NUM_POS0=35
NUM_SEED=123

### loop init
TXT_EXE=dinuc_shuffle
TXT_BATCH_SET="top"
LST_TAGS=(top01k top10k)
LST_JOBS=()

### set I/O directories
TXT_FOLDER=${TXT_BATCH_SET}
FD_BATCH="${FD_RES}/analysis_variant_motif_richard/batches_${TXT_FOLDER}_ori"
FD_DINUC="${FD_RES}/analysis_variant_motif_richard/batches_${TXT_FOLDER}_dinuc"
mkdir -p "${FD_DINUC}"

### Loop through I/O
for TXT_TAG in ${LST_TAGS[@]}; do

    ### set log
    TXT_JOB="${TXT_EXE}.${TXT_FOLDER}.${TXT_TAG}"
    FN_LOG="run.${TXT_JOB}.txt"
    FP_LOG=${FD_LOG}/${FN_LOG}
    
    ### input/output
    FP_INP="${FD_BATCH}/${FN_PREFIX}.${TXT_TAG}.ref.fa.gz"
    FP_OUT="${FD_DINUC}/${FN_PREFIX}.${TXT_TAG}.ref.fa.gz"

    ### check file existence
    if [[ ! -f "${FP_INP}" ]]; then
        echo "Missing input: ${FP_INP}"
        continue
    fi

    ### execute
    JOBID=$(sbatch \
        "${LST_OPTS[@]}" \
        --job-name="${TXT_JOB}" \
        --output="${FP_LOG}" \
        "${FP_EXE}" "${FP_INP}" "${FP_OUT}" "${NUM_POS0}" "${NUM_SEED}" \
        | tr -cd '0-9'
    )

    echo "Submitted ${TXT_TAG}: ${JOBID}"
    echo "LOG: \${FD_LOG}/${FN_LOG}"
    LST_JOBS+=("${JOBID}")
done

Submitted top01k: 43778898
LOG: ${FD_LOG}/run.dinuc_shuffle.top.top01k.txt
Submitted top10k: 43778899
LOG: ${FD_LOG}/run.dinuc_shuffle.top.top10k.txt


## Review

In [7]:
sacct_summary.sh "${LST_JOBS[@]}"

Detected multiple job IDs (2). Running batch summary.
===== Summary (.ba tasks) =====
JobID             State ExitCode ElapsedRaw   TotalCPU     MaxRSS                       NodeList 
------------ ---------- -------- ---------- ---------- ---------- ------------------------------ 
43778898.ba+  COMPLETED      0:0         11  00:00.653     41760K                 dcc-biostat-03 
43778899.ba+  COMPLETED      0:0         13  00:02.295     48356K                 dcc-biostat-03 

===== Means over .ba tasks =====
n=2  mean ElapsedRaw=12.0 sec (0.20 min)  mean MaxRSS=0.04 GiB


### Check log files

In [8]:
cat ${FD_LOG}/run.dinuc_shuffle.top.top01k.txt

Hostname:           dcc-biostat-03
Slurm Array Index:  NA
Time Stamp:         02-26-26+14:04:54
PWD:                /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts

Done. Wrote 1000 records to: /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/batches_top_dinuc/variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.top01k.ref.fa.gz

Done!
Run Time: 0 seconds



In [9]:
cat ${FD_LOG}/run.dinuc_shuffle.top.top10k.txt

Hostname:           dcc-biostat-03
Slurm Array Index:  NA
Time Stamp:         02-26-26+14:04:54
PWD:                /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts

Processed 10000 sequences...
Done. Wrote 10000 records to: /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/batches_top_dinuc/variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.top10k.ref.fa.gz

Done!
Run Time: 2 seconds



### Check output files

In [10]:
FNAME=variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.top01k.ref.fa.gz
FPATH=${FD_BATCH}/${FNAME}
zcat ${FPATH} | head -20
echo
FPATH=${FD_DINUC}/${FNAME}
zcat ${FPATH} | head -20

>chr2:234143291:C:C:G
AACTCAAAACCATCACCAACCGATTGCCATAGCTTCCATCATCTCCCTCCAAGGCCCATC
TTAACAGGCCTCCGGATAAGCTCTCTGTTCTATTCTCATTCACCAC
>chr6:111031101:A:G:C
ACTTGATAAGGCAACTGGTATAATACTGGCTGATGAAATTCCCATGTTATCTCCATTCTT
AGAAAATGAGAGGAGTAGCCATCATGATTGAGAAATAGGGCTCCCG
>chr5:73091405:T:C:A
CATCCCATTCCCACCTCTGTTAGACTAAATGATGCTAGCTGCTGCAATAGCTAAGCACAA
TACTTACTACATTGCTCTTATCACAGTCATAAGTAGGTTTGAGGAG
>chr2:37427187:G:G:T
GGGGATTCAGAGTTAGAGTGAGACCAGCAGTAAGAGGATGCAATGCCAAGGCAAAGATCA
GGACACTATCCAATCTGTATGCCACAATGGGTATGTAAACCAAAAA
>chr6:27904376:G:C:A
TGGACTATATAGCCCTAAGTGATACAGATGATGCAGTACCATGAATGATGGCCCTTCCCA
AAAGATATGTCTACGTGATAATCACTGGAATCTGCAAATGTTACCT
>chr6:123785249:A:C:G
CTGGGTTTGGGCGTTGAAGCAGCAGAAGCATAAATAATGCAATGACCTCTAGCATGACTC
AGAACCATAGACATAAGACCAGCATCTTTTGGTAGGGTGTATTCCA
>chr7:22395647:C:C:A
CAGTCTCTGTGTACATTCCCAAATGGGATGATGCACCTGTAACCCAATCAACTAAGAAAC

>chr2:234143291:C:C:G
AAACCTATGCCGCCCATCATTCACAGAAACAACTTCCCACTTATTCTCCAGCATTACCCC
CATCTGTCTCTCTCGGAGGGCTTCACCTCAATCATATCAAAGCCTC
>chr6:1110

In [11]:
FNAME=variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.top10k.ref.fa.gz
FPATH=${FD_BATCH}/${FNAME}
zcat ${FPATH} | head -20
echo
FPATH=${FD_DINUC}/${FNAME}
zcat ${FPATH} | head -20

>chr2:234143291:C:C:G
AACTCAAAACCATCACCAACCGATTGCCATAGCTTCCATCATCTCCCTCCAAGGCCCATC
TTAACAGGCCTCCGGATAAGCTCTCTGTTCTATTCTCATTCACCAC
>chr6:111031101:A:G:C
ACTTGATAAGGCAACTGGTATAATACTGGCTGATGAAATTCCCATGTTATCTCCATTCTT
AGAAAATGAGAGGAGTAGCCATCATGATTGAGAAATAGGGCTCCCG
>chr5:73091405:T:C:A
CATCCCATTCCCACCTCTGTTAGACTAAATGATGCTAGCTGCTGCAATAGCTAAGCACAA
TACTTACTACATTGCTCTTATCACAGTCATAAGTAGGTTTGAGGAG
>chr2:37427187:G:G:T
GGGGATTCAGAGTTAGAGTGAGACCAGCAGTAAGAGGATGCAATGCCAAGGCAAAGATCA
GGACACTATCCAATCTGTATGCCACAATGGGTATGTAAACCAAAAA
>chr6:27904376:G:C:A
TGGACTATATAGCCCTAAGTGATACAGATGATGCAGTACCATGAATGATGGCCCTTCCCA
AAAGATATGTCTACGTGATAATCACTGGAATCTGCAAATGTTACCT
>chr6:123785249:A:C:G
CTGGGTTTGGGCGTTGAAGCAGCAGAAGCATAAATAATGCAATGACCTCTAGCATGACTC
AGAACCATAGACATAAGACCAGCATCTTTTGGTAGGGTGTATTCCA
>chr7:22395647:C:C:A
CAGTCTCTGTGTACATTCCCAAATGGGATGATGCACCTGTAACCCAATCAACTAAGAAAC

>chr2:234143291:C:C:G
AAACCTATGCCGCCCATCATTCACAGAAACAACTTCCCACTTATTCTCCAGCATTACCCC
CATCTGTCTCTCTCGGAGGGCTTCACCTCAATCATATCAAAGCCTC
>chr6:1110